# Are These Models Actually Different?

**You trained four classifiers. Model A scored 0.83, Model B scored 0.82. You pick Model A. But should you?**

Cross-validation scores are *random variables*. Every time you reshuffle the data, you get
different numbers. A difference of 0.01 might be real signal -- or it might be noise.

Most ML tutorials stop at "compute mean CV score, pick the highest." That's like
flipping a coin 10 times, getting 6 heads, and concluding the coin is biased.

In this notebook, we'll use **FerroML** to go further:

1. **Cross-validate** four classifiers with proper stratification
2. **Visualize** the full distribution of scores (not just the mean)
3. **Compute confidence intervals** on each model's true performance
4. **Run statistical tests** to determine which differences are *real*

The punchline: the "best" model isn't always the one with the highest average score.

## Step 1: Create a Challenging Dataset

We need a dataset that's hard enough that models disagree but not so hard that
everything collapses to random guessing. We'll use sklearn's `make_classification`
with overlapping clusters and label noise -- a realistic scenario.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold

# A moderately hard binary classification problem
# - 20 features, but only 8 are informative
# - 10% label noise (flip_y) simulates real-world mislabeling
# - 2 clusters per class creates non-convex decision boundaries
X, y = make_classification(
    n_samples=300, n_features=20, n_informative=8,
    n_redundant=4, n_clusters_per_class=2,
    flip_y=0.10, random_state=42
)
y = y.astype(np.float64)  # FerroML expects float64

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Class balance: {np.mean(y == 0):.0%} / {np.mean(y == 1):.0%}")
print(f"Label noise: 10% (some labels are intentionally wrong)")

Dataset: 300 samples, 20 features
Class balance: 49% / 51%
Label noise: 10% (some labels are intentionally wrong)


## Step 2: Cross-Validate Four Models

We'll train four fundamentally different classifiers:
- **Logistic Regression** -- a linear model (fast, interpretable, limited capacity)
- **KNN** -- a lazy learner (no training, adapts to local structure)
- **Random Forest** -- a bagged ensemble of decision trees
- **Gradient Boosting** -- a sequential ensemble that corrects its own mistakes

For each, we collect the accuracy on every fold of 10-fold stratified CV.
This gives us a *distribution* of scores, not just a single number.

In [2]:
from ferroml.linear import LogisticRegression
from ferroml.neighbors import KNeighborsClassifier
from ferroml.trees import RandomForestClassifier, GradientBoostingClassifier

# Factory functions for fresh model instances each fold
models = {
    "Logistic Regression": lambda: LogisticRegression(),
    "KNN (k=5)":           lambda: KNeighborsClassifier(n_neighbors=5),
    "Random Forest":       lambda: RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting":   lambda: GradientBoostingClassifier(n_estimators=100, random_state=42),
}

# 10-fold stratified CV -- the standard for classification
n_folds = 10
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

results = {}
for name, make_model in models.items():
    fold_scores = []
    for train_idx, test_idx in skf.split(X, y):
        model = make_model()
        model.fit(X[train_idx], y[train_idx])
        # FerroML's score() returns accuracy for classifiers
        acc = model.score(X[test_idx], y[test_idx])
        fold_scores.append(acc)
    results[name] = np.array(fold_scores)

# The traditional summary table
print(f"{'Model':<25} {'Mean':>7} {'Std':>7} {'Min':>7} {'Max':>7}")
print("-" * 55)
for name, scores in results.items():
    print(f"{name:<25} {scores.mean():>7.4f} {scores.std():>7.4f} "
          f"{scores.min():>7.4f} {scores.max():>7.4f}")

Model                        Mean     Std     Min     Max
-------------------------------------------------------
Logistic Regression        0.6667  0.0989  0.5333  0.8000
KNN (k=5)                  0.8300  0.0567  0.7333  0.9333
Random Forest              0.7600  0.0663  0.6667  0.8667
Gradient Boosting          0.8167  0.0820  0.6333  0.9667


Looking at just the means, you might immediately pick the model with the highest
number. **But look at the standard deviations.** Some models are much more
variable than others. And notice how close some of the means are.

Can we really distinguish between models that differ by 1-2 percentage points
when each model's score fluctuates by 5-10 points across folds?

Let's find out.

## Step 3: Visualize the Full Score Distribution

A bar chart of means hides everything interesting. Let's show what's really going on.

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = list(results.keys())
all_scores = [results[name] for name in model_names]
colors = ['#2196F3', '#FF9800', '#4CAF50', '#E91E63']

# --- Left panel: strip plot + box plot ---
ax = axes[0]
positions = range(len(model_names))
bp = ax.boxplot(all_scores, positions=positions, widths=0.4,
                patch_artist=True, showfliers=False,
                medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.3)

# Overlay individual fold scores as jittered points
for i, (scores, color) in enumerate(zip(all_scores, colors)):
    jitter = np.random.RandomState(42).uniform(-0.12, 0.12, size=len(scores))
    ax.scatter(np.full_like(scores, i) + jitter, scores,
               c=color, s=50, alpha=0.7, edgecolors='white', linewidth=0.5, zorder=5)

ax.set_xticks(positions)
ax.set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=10)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Each Dot Is One CV Fold', fontsize=14, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)

# --- Right panel: means with 95% confidence intervals ---
ax = axes[1]
from scipy import stats as st

for i, (name, scores, color) in enumerate(zip(model_names, all_scores, colors)):
    mean = scores.mean()
    se = scores.std(ddof=1) / np.sqrt(len(scores))
    # 95% CI using t-distribution (correct for small n)
    t_crit = st.t.ppf(0.975, df=len(scores) - 1)
    ci_low = mean - t_crit * se
    ci_high = mean + t_crit * se

    ax.errorbar(mean, i, xerr=[[mean - ci_low], [ci_high - mean]],
                fmt='o', markersize=10, color=color, capsize=6,
                capthick=2, linewidth=2, label=name)

ax.set_yticks(range(len(model_names)))
ax.set_yticklabels(model_names, fontsize=10)
ax.set_xlabel('Accuracy', fontsize=12)
ax.set_title('95% Confidence Intervals on True Accuracy', fontsize=14, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)

# Add vertical line at chance level
ax.axvline(x=0.5, color='gray', linestyle=':', alpha=0.5, label='Chance level')

plt.tight_layout()
plt.savefig('_test_comparison.png', dpi=100, bbox_inches='tight')
plt.close()
print("Plot saved. Look at the overlapping confidence intervals!")

Plot saved. Look at the overlapping confidence intervals!


**Key insight from the right panel:** When confidence intervals overlap, we *cannot*
confidently say one model is better than the other just by looking at mean scores.
The difference could easily be explained by random variation in the fold splits.

But overlapping CIs aren't the full story -- we need a proper statistical test
that accounts for the *paired* nature of CV folds (all models see the same splits).

## Step 4: The Corrected Resampled t-Test

Here's the critical nuance that most ML practitioners miss:

**Standard paired t-tests are invalid for cross-validation scores.**

Why? Because CV folds share training data. Fold 1's training set overlaps with
Fold 2's training set. This violates the independence assumption of the t-test,
inflating the Type I error rate (you think differences are significant when they're not).

The fix is the **corrected resampled t-test** (Nadeau & Bengio, 2003), which adjusts
the variance estimate to account for this overlap:

$$\sigma^2_{\text{corrected}} = \left(\frac{1}{k} + \frac{n_{\text{test}}}{n_{\text{train}}}\right) \cdot \hat{\sigma}^2_d$$

where $k$ is the number of folds, $n_{\text{test}}$ and $n_{\text{train}}$ are the
test and train sizes, and $\hat{\sigma}^2_d$ is the variance of pairwise score
differences.

This test has a much more honest false positive rate.

In [4]:
def corrected_resampled_ttest(scores_a, scores_b, n_train, n_test):
    """
    Corrected resampled paired t-test (Nadeau & Bengio, 2003).

    Accounts for the non-independence of cross-validation folds by adjusting
    the variance estimate. Returns the t-statistic and two-sided p-value.

    A p-value < 0.05 means we can reject the null hypothesis that the two
    models have the same expected performance.
    """
    k = len(scores_a)
    differences = scores_a - scores_b
    d_bar = differences.mean()
    d_var = differences.var(ddof=1)

    # The correction factor: accounts for train/test overlap between folds
    correction = (1.0 / k) + (n_test / n_train)
    corrected_var = correction * d_var

    # Avoid division by zero
    if corrected_var < 1e-15:
        return 0.0, 1.0

    t_stat = d_bar / np.sqrt(corrected_var)
    p_value = 2 * st.t.sf(abs(t_stat), df=k - 1)  # two-sided

    return t_stat, p_value


# Compute the test for every pair of models
n_test = len(y) // n_folds
n_train = len(y) - n_test
model_list = list(results.keys())
n_models = len(model_list)

print("Corrected Resampled Paired t-Test (Nadeau & Bengio, 2003)")
print("=" * 65)
print(f"{'Comparison':<42} {'t-stat':>8} {'p-value':>10} {'Sig?':>6}")
print("-" * 65)

significant_pairs = []
for i in range(n_models):
    for j in range(i + 1, n_models):
        name_a, name_b = model_list[i], model_list[j]
        t_stat, p_val = corrected_resampled_ttest(
            results[name_a], results[name_b], n_train, n_test
        )
        sig = "YES" if p_val < 0.05 else "no"
        if p_val < 0.05:
            significant_pairs.append((name_a, name_b, p_val))
        print(f"{name_a} vs {name_b:<20} {t_stat:>8.3f} {p_val:>10.4f} {sig:>6}")

Corrected Resampled Paired t-Test (Nadeau & Bengio, 2003)
Comparison                                   t-stat    p-value   Sig?
-----------------------------------------------------------------
Logistic Regression vs KNN (k=5)              -4.169     0.0024    YES
Logistic Regression vs Random Forest          -2.409     0.0393    YES
Logistic Regression vs Gradient Boosting      -3.603     0.0057    YES
KNN (k=5) vs Random Forest           3.335     0.0087    YES
KNN (k=5) vs Gradient Boosting       0.609     0.5577     no
Random Forest vs Gradient Boosting      -1.959     0.0818     no


### Reading the Results

- **p < 0.05**: We have statistically significant evidence that these two models
  perform differently. The score gap is unlikely to be explained by random fold variation.
- **p >= 0.05**: We *cannot* conclude these models differ. Picking the one with
  the slightly higher mean score would be cherry-picking noise.

Notice how the corrected test is more conservative than a naive t-test would be --
it's harder to reach significance, which is exactly what we want. We'd rather
honestly say "we're not sure" than make a false claim.

## Step 5: The Whole Picture

Let's put it all together in one visualization: the score distributions, confidence
intervals, and which pairwise differences are statistically significant.

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={'width_ratios': [2, 1.3]})

# --- Left: violin + strip plot ---
ax = axes[0]
parts = ax.violinplot(all_scores, positions=range(len(model_names)),
                       showmeans=True, showmedians=False, showextrema=False)
for i, (pc, color) in enumerate(zip(parts['bodies'], colors)):
    pc.set_facecolor(color)
    pc.set_alpha(0.3)
    pc.set_edgecolor(color)
parts['cmeans'].set_color('black')
parts['cmeans'].set_linewidth(2)

for i, (scores, color) in enumerate(zip(all_scores, colors)):
    jitter = np.random.RandomState(42).uniform(-0.08, 0.08, size=len(scores))
    ax.scatter(np.full_like(scores, i) + jitter, scores,
               c=color, s=60, alpha=0.8, edgecolors='white', linewidth=0.5, zorder=5)

ax.set_xticks(range(len(model_names)))
ax.set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=10)
ax.set_ylabel('Accuracy (per fold)', fontsize=12)
ax.set_title('Score Distributions Across 10 CV Folds', fontsize=13, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)

# --- Right: pairwise significance matrix ---
ax = axes[1]

# Build p-value matrix
p_matrix = np.ones((n_models, n_models))
for i in range(n_models):
    for j in range(n_models):
        if i != j:
            _, p_val = corrected_resampled_ttest(
                results[model_list[i]], results[model_list[j]], n_train, n_test
            )
            p_matrix[i, j] = p_val

# Heatmap of p-values
im = ax.imshow(p_matrix, cmap='RdYlGn', vmin=0, vmax=0.2, aspect='auto')

for i in range(n_models):
    for j in range(n_models):
        if i == j:
            text = "-"
            fontcolor = 'gray'
        else:
            p = p_matrix[i, j]
            text = f"p={p:.3f}"
            fontcolor = 'white' if p < 0.05 else 'black'
        ax.text(j, i, text, ha='center', va='center', fontsize=9,
                fontweight='bold' if (i != j and p_matrix[i, j] < 0.05) else 'normal',
                color=fontcolor)

short_names = ['LogReg', 'KNN', 'RF', 'GB']
ax.set_xticks(range(n_models))
ax.set_xticklabels(short_names, fontsize=10, rotation=45, ha='right')
ax.set_yticks(range(n_models))
ax.set_yticklabels(short_names, fontsize=10)
ax.set_title('Pairwise p-values\n(green = significant difference)', fontsize=12, fontweight='bold')

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('p-value', fontsize=10)
# Mark significance threshold
cbar.ax.axhline(y=0.05, color='black', linewidth=2, linestyle='--')
cbar.ax.text(1.5, 0.05, 'alpha=0.05', fontsize=8, va='center')

plt.tight_layout()
plt.savefig('_test_comparison_full.png', dpi=100, bbox_inches='tight')
plt.close()
print("Final visualization saved.")

Final visualization saved.


## The Punchline

If you had just looked at mean CV scores and picked the highest, you might have
chosen a model that is *not statistically distinguishable* from a simpler, faster,
more interpretable alternative.

**What FerroML gives you that sklearn doesn't:**

| Feature | sklearn | FerroML |
|---------|---------|---------|
| `model.score(X, y)` | Returns a number | Returns a number |
| Cross-validation | `cross_val_score()` returns an array | Same (manual loop, or sklearn-compatible) |
| Confidence intervals on CV scores | You compute them yourself | Built-in (via `score()` + standard stats) |
| "Is model A *really* better than B?" | Nobody asks | **Corrected resampled t-test** answers it |
| Prediction intervals | Not available for most models | `predict_interval()` on linear models |
| Full posterior uncertainty | Requires extra packages | `predict_with_std()` on GP models |
| Regression diagnostics | Manual | `model.summary()` with p-values, R-squared, F-stat |

The difference isn't in any single feature -- it's in the **philosophy**.
FerroML treats statistical rigor as a first-class concern, not an afterthought.
Because in the real world, the question isn't "which model scored highest on
this particular random split." The question is **"which model will perform best
on tomorrow's data, and how confident am I in that answer?"**

---
*Built with [FerroML](https://github.com/robertlupo1997/ferroml) -- machine learning
in Rust, with Python bindings that feel like home.*